## Notebook14b

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c

import torch
import torch.nn as nn
import torch.optim as optim

theme_set(theme_minimal())
pl.Config(tbl_rows=25)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
!mkdir -p models
!wget -q -nc -P models https://humanitiesdata.org/models/model_cnn_agnews.pt
!wget -q -nc -P models https://humanitiesdata.org/models/model_pre_agnews.pt
!wget -q -nc -P models https://humanitiesdata.org/models/word2vec.model

In [ ]:
agnews = pl.read_parquet(ub + "data/agnews_pca.parquet")

### Part 1: Word Embedding

1. We will start by installing the gensim library, which is not one of the libraries available by default on Colab.

In [ ]:
!pip install gensim

2. Once the library is install, load the `Word2Vec` model.

In [ ]:
from gensim.models import Word2Vec

3. As we did in the text with the IMDb data, we want to compute the tokens in the AG News dataset.

In [ ]:
agnews = (
    agnews
    .with_columns(
        tokens = c.text.str.to_lowercase().str.extract_all(r"[a-z]+")
    )
)
agnews

4. Next, we build the Word2Vec model. Here, we use the same settings as in the text except that we increase the minimum word count due to the increased size of the texts. The code here will only build the model if it has not already been run to save some time in future runs.

In [ ]:
try:
    model = Word2Vec.load("models/word2vec.model")
except FileNotFoundError:
    model = Word2Vec(
        sentences=agnews["tokens"].to_list(),
        vector_size=100,
        window=2,
        min_count=25,
        sg=1,
        epochs=20
    )
    model.save("models/word2vec.model")

5. Check the length of the embedding. These are the number of words in the text.

In [ ]:
len(model.wv)

6. Now that we have the embedding, we will apply it to all of the texts.

In [ ]:
embed = pl.DataFrame({
    "word": list(set(model.wv.key_to_index)),
    "embedding": [model.wv[w].tolist() for w in set(model.wv.key_to_index)],
})
embed

7. And finally, we can check how the embedding works by seeing what words are similar to one another. Start with my intial suggestion, but play around with different word choices.

In [ ]:
model.wv.most_similar("good", topn=10)

### Part 2: CNNs

8. Now let's create the training and testing data for our AG News corpus.

In [ ]:
X, X_train, X_test, y, y_train, y_test, cn = DSTorch.load_text(
    df=agnews,
    model=model,
    tokens_expr=c.tokens,
    label_expr=c.label
)
embedding_dim = model.vector_size
embedding_matrix = torch.tensor(model.wv.vectors, dtype=torch.float32)
embedding_matrix.shape

9. Here is the Torch CNN model that we will use to train the AG News corpus to detect the category of an article.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, embedding_matrix, num_classes, num_filters=100, filter_size=3):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, _freeze=True)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = False
        
        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=num_filters,
            kernel_size=filter_size,
            padding=filter_size // 2
        )
        self.relu = nn.ReLU()
        
        self.classifier = nn.Sequential(
            nn.Linear(num_filters, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.permute(0, 2, 1)
        conv_out = self.relu(self.conv(embedded))
        pooled = conv_out.max(dim=2)[0]
        
        output = self.classifier(pooled)
        return output

10. And then we create an instance of the model.

In [ ]:
model_cnn = TextCNN(
    embedding_matrix,
    num_classes=len(cn),
    num_filters=25,
    filter_size=3
)
optimizer = optim.Adam(model_cnn.parameters(), lr=0.0003)

11. Finally, train the data. If the model already exists, just read in the weights. This saves time on re-runs.

In [ ]:
try:
    model_cnn.load_state_dict(torch.load("models/model_cnn_agnews.pt"))
except FileNotFoundError:
    DSTorch.train(model_cnn, optimizer, X_train, y_train, num_epochs=10)
    torch.save(model_cnn.state_dict(), "models/model_cnn_agnews.pt")

12. Here are the training and validation error rates:

In [ ]:
DSTorch.score_text(model_cnn, X_train, y_train)

In [ ]:
DSTorch.score_text(model_cnn, X_test, y_test)

13. As well as the confusion matrix:

In [ ]:
DSTorch.confusion_matrix(model_cnn, X_test, y_test, cn)

### Part 3: Precomputed Embeddings

14. We saw in the text the precomputed embeddings are faster and often more accurate for downstream models. Look at all of the available models from gensim:

In [ ]:
import gensim.downloader as api

print(list(api.info()['models'].keys()))

15. We will load the `glove-wiki-gigaword-100` model here. It needs to download the first time, so it may take a few moments to finish.

In [ ]:
pretrained = api.load("glove-wiki-gigaword-100")

16. Find the closest words to banana, orange, and apple. Notice that these are not context specific, leading to some inconsistent results in the three fruit names.

In [ ]:
pretrained.most_similar("banana", topn=10)

In [ ]:
pretrained.most_similar("orange", topn=10)

In [ ]:
pretrained.most_similar("apple", topn=10)

17. Try your own words as well and see what other terms the model places it next to.

In [ ]:
pretrained.most_similar("good", topn=10)

### Part 4: Refitting with Pretrained Embeddings

18. In this final section, we will embed the AG News data using the pretrained embeddings. Here is the code to match the words with the available vocabulary.

In [ ]:
vocab = list(model.wv.key_to_index.keys())
embedding_dim = pretrained.vector_size

pretrained_matrix = np.zeros((len(vocab), embedding_dim))
found_count = 0

for i, word in enumerate(vocab):
    if word in pretrained:
        pretrained_matrix[i] = pretrained[word]
        found_count += 1
    else:
        pretrained_matrix[i] = np.random.uniform(-0.25, 0.25, embedding_dim)

print(f"Found {found_count} of {len(vocab)} words in pre-trained embeddings")
pretrained_matrix = torch.tensor(pretrained_matrix, dtype=torch.float32)

19. And again, we can create a similar model.

In [ ]:
model_pretrained = TextCNN(
    pretrained_matrix,
    num_classes=len(cn),
    num_filters=25,
    filter_size=3
)
optimizer_pretrained = optim.Adam(model_pretrained.parameters(), lr=0.0003)

20. And train it, using the saved weights if they are available.

In [ ]:
try:
    model_pretrained.load_state_dict(torch.load("models/model_pre_agnews.pt"))
except FileNotFoundError:
    DSTorch.train(model_pretrained, optimizer, X_train, y_train, num_epochs=10)
    torch.save(model_pretrained.state_dict(), "models/model_pre_agnews.pt")

21. Compare the error rates to the embeddings we trained ourselves:

In [ ]:
DSTorch.score_text(model_cnn, X_train, y_train)

In [ ]:
DSTorch.score_text(model_cnn, X_test, y_test)

22. As well as the confusion matrix to the previous model.

In [ ]:
DSTorch.confusion_matrix(model_cnn, X_test, y_test, cn)